In [1]:
#%pip install pandas psycopg[binary] python-dotenv

In [5]:
import os, re
from datetime import date 
from pathlib import Path

import pandas as pd
import psycopg
from dotenv import load_dotenv

In [6]:
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "docker-compose.yaml").exists())
DATA = ROOT / "data"                 
SQL  = ROOT / "sql"

load_dotenv(ROOT/".env")

PG_CONFIG = {
    "dbname": os.environ["POSTGRES_DB"],
    "user": os.environ["POSTGRES_USER"],
    "password": os.environ["POSTGRES_PASSWORD"],
    "host": "127.0.0.1",     
    "port": 5433,            
}

try:
    conn = psycopg.connect(**PG_CONFIG)
    print(conn.execute("SELECT version()").fetchone())
except Exception as e:
    print(f"Error de conexión: {e}")
    raise

('PostgreSQL 18.6 (Debian 18.6-1.pgdg13+2) on x86_64-pc-linux-gnu, compiled by gcc (Debian 14.2.0-19) 14.2.0, 64-bit',)


In [7]:
RESET = False

if RESET:
    conn.execute("DROP SCHEMA public CASCADE")
    conn.execute("CREATE SCHEMA public")
    print("Esquema reiniciado")

conn.execute((SQL / "ddl.sql").read_text(encoding="utf-8"))
conn.commit()
print("DDL ejecutado")

DDL ejecutado


In [8]:
def normalizar_fecha(valor):
    """
    Convierte fechas parciales o con hora a 'YYYY-MM-DD' (válido para DATE en PostgreSQL).
      '2027'                       -> '2027-01-01'   (mes/día faltantes = 01)
      '2027-04'                    -> '2027-04-01'
      '2001-04-03T00:00:00+00:00'  -> '2001-04-03'
      '2023-02-30' (día inválido)  -> '2023-02-01'
      None / NaN / '' / 'Unknown'  -> None
    """
    if valor is None or (not isinstance(valor, str) and pd.isna(valor)):
        return None
    m = re.match(r"^\s*(\d{4})(?!\d)(?:-(\d{1,2})(?:-(\d{1,2}))?)?", str(valor))
    if not m:
        return None
    anio, mes, dia = int(m.group(1)), int(m.group(2) or 1), int(m.group(3) or 1)
    for candidato in ((anio, mes, dia), (anio, mes, 1), (anio, 1, 1)):    # se degrada hasta una fecha válida
        try:
            return date(*candidato).isoformat()
        except ValueError:
            continue
    return None

def clean(v):
    """NaN/NaT -> None; numpy -> tipos nativos; 
    1998.0 -> 1998 (columnas enteras que pandas convirtió a float)."""
    if pd.isna(v):
        return None
    if hasattr(v, "item"):
        v = v.item()
    if isinstance(v, float) and v.is_integer():
        v = int(v)
    return v

def upsert(cur, table, df, pk):
    """01_extraction.ipynb: upsert() - Inserta o actualiza filas en la tabla de destino según la clave primaria."""
    cols = list(df.columns)
    non_pk = [c for c in cols if c not in pk]
    if non_pk:
        # ON CONFLICT (pk) DO UPDATE SET col = EXCLUDED.col, ... actualiza las columnas no PK con los valores de la fila que causó el conflicto
        conflict = f"ON CONFLICT ({', '.join(pk)}) DO UPDATE SET " + ", ".join(f"{c} = EXCLUDED.{c}" for c in non_pk)
    else:
        # ON CONFLICT (pk) DO NOTHING evita errores de duplicado si la fila ya existe y no hay columnas no PK para actualizar
        conflict = f"ON CONFLICT ({', '.join(pk)}) DO NOTHING"
    sql = f"INSERT INTO {table} ({', '.join(cols)}) VALUES ({', '.join(['%s'] * len(cols))}) {conflict}"
    rows = [tuple(clean(v) for v in r) for r in df.itertuples(index=False, name=None)]
    cur.executemany(sql, rows)
    return len(rows)

# (tabla, clave primaria) en orden de carga
LOAD_ORDER = [
    ("genre", ["genre_id"]),
    ("demographic", ["demographic_id"]),
    ("theme", ["theme_id"]),
    ("producer", ["producer_id"]),
    ("person", ["person_id"]),
    ("anime", ["anime_id"]),
    ("anime_genre", ["anime_id", "genre_id"]),
    ("anime_demographic", ["anime_id", "demographic_id"]),
    ("anime_theme", ["anime_id", "theme_id"]),
    ("anime_producer", ["anime_id", "producer_id", "role"]),
    ("anime_staff", ["anime_id", "person_id", "position"]),
    ("anime_relation", ["anime_id", "related_anime_id", "relation_type"]),
    ("anime_recommendation", ["anime_id", "recommended_anime_id"]),
]

with conn.cursor() as cur:
    for table, pk in LOAD_ORDER:
        # keep_default_na=False: solo la cadena vacía es NULL (evita que un título como "NA" o "None" se pierda)
        df = pd.read_csv(DATA / f"{table}.csv", keep_default_na=False, na_values=[""])
        if table == "anime":
            df["aired_from"] = df["aired_from"].map(normalizar_fecha)
        n = upsert(cur, table, df, pk)
        print(f"{table} {n} filas")
conn.commit()

genre 21 filas
demographic 5 filas
theme 52 filas
producer 1285 filas
person 22242 filas
anime 7359 filas
anime_genre 13681 filas
anime_demographic 2885 filas
anime_theme 9060 filas
anime_producer 22722 filas
anime_staff 328563 filas
anime_relation 15992 filas
anime_recommendation 19845 filas


In [9]:
def query_df(sql, params=None):
    with conn.cursor() as cur:
        cur.execute(sql, params)
        return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


tablas = [t for t, _ in LOAD_ORDER]
conteos = {t: conn.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0] for t in tablas}
pd.Series(conteos, name="filas").to_frame()

,filas
genre,21
demographic,5
theme,52
producer,1285
person,22242
anime,7359
anime_genre,13681
anime_demographic,2885
anime_theme,9060
anime_producer,22722


In [10]:
query_df("""
    SELECT title, score 
    FROM ANIME 
    WHERE SCORE IS NOT NULL
    ORDER BY SCORE DESC LIMIT 100""")

,title,score
0,Sousou no Frieren,9.24
1,Fullmetal Alchemist: Brotherhood,9.11
2,Steins;Gate,9.07
3,Shingeki no Kyojin Season 3 Part 2,9.06
4,Re:Zero kara Hajimeru Isekai Seikatsu 4th Season,9.06
...,...,...
95,Ping Pong the Animation,8.63
96,Fate/stay night Movie: Heaven's Feel - III. Sp...,8.63
97,Mo Dao Zu Shi: Wanjie Pian,8.63
98,Chiikawa,8.63
